[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/apis-avanzadas/03-files-api.ipynb)

# Files API: Documentos Persistentes en la API

> Aprende a subir documentos una vez y reutilizarlos en múltiples llamadas,
> reduciendo latencia y preparando el terreno para Prompt Caching.

## Objetivos
- Subir archivos de texto, CSV y PDF a la API
- Referenciar archivos por ID en llamadas posteriores
- Construir una base de conocimiento compartida
- Gestionar el ciclo de vida de los archivos

## 1. Instalación de dependencias

In [ ]:
%pip install anthropic pandas --quiet

## 2. Subir documentos de texto

In [ ]:
import anthropic
import io
import pandas as pd

client = anthropic.Anthropic()

# Crear documentos de ejemplo en memoria
MANUAL_PRODUCTO = """
MANUAL DE USUARIO — ProductoX v2.5

1. INSTALACIÓN
   Requisitos mínimos: Python 3.9+, 4GB RAM, 2GB disco.
   Ejecutar: pip install productox && productox --init

2. CONFIGURACIÓN BÁSICA
   Editar config.yaml: 
     api_key: TU_API_KEY
     max_conexiones: 10
     timeout: 30

3. MODO AVANZADO
   Activar con: productox --advanced-mode on
   Permite: clustering, caché distribuido, métricas en tiempo real.

4. ERRORES COMUNES
   Error 401: API key inválida. Verifica config.yaml.
   Error 503: Servidor no disponible. Revisar timeout.
   Error 429: Rate limit. Reducir max_conexiones.

5. SOPORTE
   Email: soporte@productox.com | Tel: 900 123 456 (L-V 9-18h)
"""

FAQ_TEXTO = """
PREGUNTAS FRECUENTES — ProductoX

P: ¿Puedo usar ProductoX sin conexión?
R: No, requiere conexión a internet para validar la licencia.

P: ¿Cuántos dispositivos incluye mi licencia?
R: Plan Básico: 1 dispositivo. Plan Pro: 5. Enterprise: ilimitado.

P: ¿Cómo exporto mis datos?
R: Desde el menú Archivo → Exportar → elegir formato (CSV, JSON, Excel).

P: ¿Hay versión de prueba?
R: Sí, 14 días gratuitos sin tarjeta de crédito en productox.com/trial
"""

# Subir ambos documentos
print("Subiendo documentos...")
archivo_manual = client.beta.files.upload(
    file=("manual.txt", io.BytesIO(MANUAL_PRODUCTO.encode()), "text/plain")
)
archivo_faq = client.beta.files.upload(
    file=("faq.txt", io.BytesIO(FAQ_TEXTO.encode()), "text/plain")
)

print(f"✓ Manual subido: {archivo_manual.id}")
print(f"✓ FAQ subido:    {archivo_faq.id}")
print(f"  Tamaño manual: {archivo_manual.size} bytes")
print(f"  Tamaño FAQ:    {archivo_faq.size} bytes")

## 3. Usar archivos en llamadas a la API

In [ ]:
def preguntar_base_conocimiento(pregunta: str) -> str:
    """Responde usando los documentos subidos previamente."""
    resp = client.beta.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=400,
        messages=[{
            "role": "user",
            "content": [
                {
                    "type": "document",
                    "source": {"type": "file", "file_id": archivo_manual.id}
                },
                {
                    "type": "document",
                    "source": {"type": "file", "file_id": archivo_faq.id}
                },
                {
                    "type": "text",
                    "text": f"Basándote en los documentos, responde: {pregunta}"
                }
            ]
        }],
        betas=["files-api-2025-04-14"]
    )
    return resp.content[0].text

preguntas = [
    "¿Cuáles son los requisitos mínimos del sistema?",
    "¿Cómo activo el modo avanzado?",
    "¿Qué hago si me aparece el error 429?",
    "¿Puedo usar ProductoX en 3 ordenadores con el plan Pro?",
]

print("=== BASE DE CONOCIMIENTO CON FILES API ===")
for pregunta in preguntas:
    respuesta = preguntar_base_conocimiento(pregunta)
    print(f"\n❓ {pregunta}")
    print(f"   {respuesta[:200]}")

## 4. Subir CSV y analizar con IA

In [ ]:
# Crear CSV de ventas
import io
df_ventas = pd.DataFrame({
    "mes": ["Enero", "Febrero", "Marzo", "Abril", "Mayo", "Junio"],
    "ventas": [45200, 38100, 52300, 49800, 61200, 58900],
    "clientes_nuevos": [23, 18, 31, 27, 38, 35],
    "churn": [2, 3, 1, 4, 2, 3],
    "region": ["Norte", "Norte", "Sur", "Sur", "Este", "Este"]
})

csv_bytes = df_ventas.to_csv(index=False).encode()
archivo_csv = client.beta.files.upload(
    file=("ventas_q1q2.csv", io.BytesIO(csv_bytes), "text/csv")
)
print(f"✓ CSV subido: {archivo_csv.id} ({archivo_csv.size} bytes)")

# Analizar CSV con múltiples preguntas sin re-subir
def analizar_csv(pregunta: str) -> str:
    resp = client.beta.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=300,
        messages=[{
            "role": "user",
            "content": [
                {"type": "document",
                 "source": {"type": "file", "file_id": archivo_csv.id}},
                {"type": "text", "text": pregunta}
            ]
        }],
        betas=["files-api-2025-04-14"]
    )
    return resp.content[0].text

print("\n=== ANÁLISIS CSV CON FILES API ===")
print(df_ventas.to_string(index=False))
print()

for pregunta in [
    "¿Cuál fue el mes con más ventas y cuánto representó sobre el total?",
    "¿Cuál es la tasa de churn media mensual?",
    "¿Qué región tuvo mejor rendimiento?",
]:
    print(f"\n❓ {pregunta}")
    print(f"   {analizar_csv(pregunta).strip()[:200]}")

## 5. Gestión del ciclo de vida de archivos

In [ ]:
# Listar todos los archivos subidos
print("=== ARCHIVOS EN LA API ===")
archivos_listados = client.beta.files.list()
for arch in archivos_listados.data:
    tamaño_kb = arch.size / 1024
    print(f"  {arch.id} | {arch.filename} | {tamaño_kb:.1f} KB")

# Obtener metadata de un archivo específico
info_manual = client.beta.files.retrieve_metadata(archivo_manual.id)
print(f"\nMetadata del manual:")
print(f"  ID: {info_manual.id}")
print(f"  Nombre: {info_manual.filename}")
print(f"  Tamaño: {info_manual.size} bytes")

# Eliminar archivo (cuando ya no se necesite)
print(f"\nEliminando CSV de prueba...")
client.beta.files.delete(archivo_csv.id)
print(f"✓ {archivo_csv.id} eliminado")

print("""
=== RESUMEN FILES API ===

Casos de uso principales:
  ✓ Base de conocimiento compartida entre usuarios
  ✓ PDFs grandes que se consultan frecuentemente
  ✓ Datasets CSV para análisis recurrentes
  ✓ Manuales técnicos en sistemas de soporte
  ✓ Combinado con Prompt Caching: máximo ahorro

Límites importantes:
  • Máximo 32 MB por archivo
  • TTL configurable (hasta 30 días por defecto)
  • Requiere beta header: files-api-2025-04-14
""")